# **Notebook: Final Report Generation – Automated Documentation**

**Project:** Population-Scale Cross-Disorder Atlas of the Human Prefrontal Cortex  
**Step:** 6. Report Assembly  
**Input:** All previously generated figures and statistical summaries.  
**Output:** Final PDF/DOCX Report (`Transcriptomics_Project_Report.docx`)

---

### **1. Introduction and Objectives**

This final notebook serves as the **assembly engine** for the project. It automates the creation of the final technical report by aggregating:
1.  **Quality Control Metrics:** Post-filtering statistics from the single-cell pipeline.
2.  **Dimensionality Reduction:** UMAP and PCA plots visualizing cell type clusters.
3.  **Differential Expression Results:** Volcano plots, heatmaps, and summary tables generated in the post-R visualization step.

**Key Features:**
* **Automated Structuring:** Generates a structured document (Introduction, Methods, Results, Discussion) using `python-docx`.
* **Dynamic Insertion:** Automatically fetches the latest versions of figures and tables, ensuring the report is always up-to-date with the analysis.
* **Formatting:** Applies consistent academic formatting (headings, captions, page breaks).

**Note:** This notebook does not perform analysis. It compiles the outputs of steps 1 through 5 into a coherent deliverable.

In [1]:
from docx import Document
from docx.shared import Pt, RGBColor, Inches, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import os
import pandas as pd

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

BASE_PATH = "C:/Z/AIDA_transcriptomics_project/transcriptomics-code"

PATHS = {
    "EDA": os.path.join(BASE_PATH, "eda_results"),
    "FIGURES": os.path.join(BASE_PATH, "figures"),
    "FIGURES_SCP": os.path.join(BASE_PATH, "figures", "single_cell_pipeline"),
    "FIGURES_RAPPORT": os.path.join(BASE_PATH, "figures_pour_rapport"),
    # Figures issues de R / Python Visualization
    "R_FIGURES": os.path.join(BASE_PATH, "Figures_Finales"),
}

# Sous-dossiers spécifiques
PATHS["R_VOLCANO"] = os.path.join(PATHS["R_FIGURES"], "volcano")
PATHS["R_HEATMAP"] = os.path.join(PATHS["R_FIGURES"], "heatmaps")
PATHS["R_BAR"] = os.path.join(PATHS["R_FIGURES"], "barplots")
PATHS["R_SUMMARY"] = os.path.join(PATHS["R_FIGURES"], "summary_DEGs.csv")

# =============================================================================
# 2. INITIALISATION DOCUMENT
# =============================================================================
doc = Document()

# Marges (2cm)
for section in doc.sections:
    section.top_margin = Cm(2)
    section.bottom_margin = Cm(2)
    section.left_margin = Cm(2.5)
    section.right_margin = Cm(2.5)

# Police par défaut et JUSTIFICATION
style = doc.styles['Normal']
style.font.name = 'Times New Roman'
style.font.size = Pt(12)
style.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY  # <--- CHANGEMENT 2: Justifié

# Configuration des titres
def set_heading_font(heading_style, size=None, bold=True):
    font = heading_style.font
    font.name = 'Times New Roman'
    font.color.rgb = RGBColor(0, 0, 0)
    font.bold = bold
    if size: font.size = Pt(size)

set_heading_font(doc.styles['Heading 1'], size=16)
set_heading_font(doc.styles['Heading 2'], size=14)
set_heading_font(doc.styles['Heading 3'], size=12)

# =============================================================================
# 3. FONCTIONS UTILITAIRES (MODIFIÉES)
# =============================================================================

def apply_note_style(doc, title, text, icon="📘", color=RGBColor(26, 84, 144)):
    """
    Applique le style spécifique pour les notes (Pedago/Tech/Methodo).
    Police: Century Gothic, Size: 9, Indent: 1 inch.
    """
    p = doc.add_paragraph()
    
    # Titre de la note
    runner = p.add_run(f"{icon} {title}\n")
    runner.bold = True
    runner.font.name = 'Century Gothic' # <--- CHANGEMENT 3
    runner.font.size = Pt(9)            # <--- CHANGEMENT 3
    runner.font.color.rgb = color
        
    # Texte de la note
    runner2 = p.add_run(text)
    runner2.font.name = 'Century Gothic' # <--- CHANGEMENT 3
    runner2.font.size = Pt(9)            # <--- CHANGEMENT 3
    runner2.font.color.rgb = color
    
    # Indentation et Espacement
    p.paragraph_format.left_indent = Inches(1) # <--- CHANGEMENT 3 (Indentation 1)
    p.paragraph_format.space_after = Pt(12)
    # On force l'alignement à gauche pour les notes (souvent plus joli que justifié sur petit texte)
    p.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.LEFT 

def add_pedagogical_note(doc, title, text):
    apply_note_style(doc, f"NOTE PÉDAGOGIQUE - {title}", text, icon="📘", color=RGBColor(26, 84, 144))

def add_technical_note(doc, title, text):
    apply_note_style(doc, f"NOTE TECHNIQUE - {title}", text, icon="⚙️", color=RGBColor(102, 102, 102))

def add_methodo_note(doc, title, text):
    apply_note_style(doc, f"NOTE MÉTHODOLOGIQUE - {title}", text, icon="📝", color=RGBColor(112, 48, 160))

def add_figure(doc, path, caption, width=Inches(6)):
    """Ajoute une figure avec recherche intelligente du fichier."""
    try:
        # Recherche flexible si le nom exact n'existe pas
        if not os.path.exists(path):
            folder = os.path.dirname(path)
            filename = os.path.basename(path)
            # Mots clés pour essayer de retrouver le fichier
            keywords = filename.replace("Volcano_", "").replace("Barplot_", "").replace(".png", "").split("_")
            if os.path.exists(folder):
                for f in os.listdir(folder):
                    # Si tous les mots clés (ex: Microglia, AD, CTRL) sont dans le nom du fichier
                    if all(k in f for k in keywords):
                        path = os.path.join(folder, f)
                        break
        
        if os.path.exists(path):
            p = doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run = p.add_run()
            run.add_picture(path, width=width)
            
            p_cap = doc.add_paragraph(caption)
            p_cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
            p_cap.runs[0].italic = True
            p_cap.runs[0].font.size = Pt(10)
            p_cap.paragraph_format.space_after = Pt(12)
        else:
            p = doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run = p.add_run(f"[IMAGE MANQUANTE]\nChemin cherché : {path}")
            run.font.color.rgb = RGBColor(255, 0, 0)
            run.bold = True
    except Exception as e:
        print(f"Erreur image {path}: {e}")

def add_table_with_style(doc, data, headers, caption):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = 'Light Grid Accent 1'
    
    hdr_cells = table.rows[0].cells
    for i, header in enumerate(headers):
        hdr_cells[i].text = header
        for p in hdr_cells[i].paragraphs:
            for run in p.runs:
                run.font.bold = True
                run.font.size = Pt(11)
    
    for row_data in data:
        row = table.add_row().cells
        for i, cell_data in enumerate(row_data):
            row[i].text = str(cell_data)
            for p in row[i].paragraphs:
                for run in p.runs:
                    run.font.size = Pt(11)
    
    p = doc.add_paragraph(caption)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.runs[0].italic = True
    p.runs[0].font.size = Pt(10)
    p.paragraph_format.space_after = Pt(12)

# =============================================================================
# 4. CONTENU DU RAPPORT (TEXTE COMPLET)
# =============================================================================

# PAGE DE GARDE (Modifiée 1)
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("FACULTÉ DES SCIENCES ET INGÉNIERIE\nSorbonne Université\n\n\n")
run.bold = True
run.font.size = Pt(14)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("\n\nAnalyse transcriptomique unicellulaire du cortex préfrontal dans les maladies neurodégénératives :\nIdentification de signatures cellulaires distinctes dans la maladie d'Alzheimer et de Parkinson\n\n")
run.bold = True
run.font.size = Pt(16)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("(Single-Cell Transcriptomic Analysis of the Prefrontal Cortex in Neurodegenerative Diseases:\nIdentifying Distinct Cellular Signatures in Alzheimer's and Parkinson's Disease)\n\n\n")
run.italic = True
run.font.size = Pt(12)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = p.add_run("Auteures :\nYara DAGHER\nElodie HUSSON\nLaïla EL BOUHALI\n\n\n")
run.bold = True
run.font.size = Pt(14)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.add_run("UE : MU5BIS01 - Analyse de données transcriptomiques à l'échelle unicellulaire\n")
p.add_run("Master Artificial Intelligence and Data Analysis (AIDA) 2025-2026\n\n\n\n")

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
p.add_run("Date de soumission : 7 décembre 2025")

doc.add_page_break()

# INTRO
doc.add_heading('1. Introduction', level=1)
doc.add_heading('1.1. Contexte biologique et clinique', level=2)
doc.add_paragraph(
    "Les maladies neurodégénératives, telles que la maladie d'Alzheimer (AD) et la maladie de Parkinson (PD), "
    "représentent un défi majeur de santé publique dans le contexte du vieillissement démographique mondial. "
    "Bien que ces pathologies présentent des manifestations cliniques distinctes — troubles mnésiques prédominants "
    "pour l'AD et symptômes moteurs pour la PD —, elles partagent des mécanismes physiopathologiques communs, "
    "notamment l'accumulation d'agrégats protéiques pathologiques (plaques amyloïdes et dégénérescences "
    "neurofibrillaires dans l'AD ; corps de Lewy contenant l'alpha-synucléine dans la PD) et une neuroinflammation "
    "chronique médiée par l'activation gliale. Le Cortex Préfrontal DorsoLatéral (DLPFC), région clé impliquée "
    "dans les fonctions exécutives, la mémoire de travail et la régulation émotionnelle, est particulièrement "
    "vulnérable dans ces troubles, avec une atrophie progressive et des altérations fonctionnelles précoces."
)

add_pedagogical_note(doc, "Pourquoi le DLPFC ?", 
    "Le Cortex Préfrontal DorsoLatéral (DLPFC - aires de Brodmann 9/46) est la zone cérébrale responsable "
    "des fonctions cognitives supérieures : mémoire de travail, planification, inhibition et prise de décision. "
    "C'est le « centre de contrôle exécutif » du cerveau. Dans l'AD, cette région subit une neurodégénérescence "
    "précoce corrélée au déclin cognitif. Dans la PD, bien que les symptômes moteurs dominent cliniquement, "
    "des dysfonctionnements cognitifs du DLPFC apparaissent fréquemment, notamment dans les formes avec démence. "
    "L'étude du DLPFC permet donc de cibler les mécanismes moléculaires conduisant au déclin cognitif commun "
    "à ces deux pathologies."
)

doc.add_heading('1.2. Apport du single-nucleus RNA sequencing (snRNA-seq)', level=2)
doc.add_paragraph(
    "Les approches transcriptomiques classiques sur tissu entier (bulk RNA-seq) moyennent le signal d'expression "
    "génique sur des millions de cellules hétérogènes, diluant ainsi la contribution spécifique des types cellulaires "
    "minoritaires mais biologiquement critiques, tels que la microglie (cellules immunitaires résidentes du cerveau) "
    "ou les astrocytes réactifs. Le séquençage RNA à résolution unicellulaire (single-cell RNA-seq, scRNA-seq) "
    "permet de profiler l'expression génique individuellement pour chaque cellule. Dans le contexte des tissus "
    "post-mortem congelés, la variante single-nucleus RNA-seq (snRNA-seq) est privilégiée car elle séquence les noyaux "
    "cellulaires plutôt que les cellules entières, contournant ainsi les problèmes de dissociation tissulaire et "
    "permettant de préserver la qualité des ARN messagers. Cette approche permet de déconvoluer l'hétérogénéité "
    "cellulaire du tissu cérébral, d'identifier les types cellulaires vulnérables à la pathologie et de révéler "
    "des signatures moléculaires spécifiques invisibles en bulk RNA-seq."
)

add_pedagogical_note(doc, "snRNA-seq vs bulk RNA-seq",
    "• Bulk RNA-seq : Moyenne le signal de millions de cellules différentes. Analogie : Une photo floue de toute "
    "une foule. Si 1% de la foule porte un t-shirt rouge (microglie), cette couleur sera statistiquement invisible, "
    "noyée dans la moyenne.\n\n"
    "• snRNA-seq : Donne le profil d'expression de chaque noyau individuel. Analogie : Des photos haute résolution "
    "de chaque personne dans la foule. On voit clairement le 1% qui porte un t-shirt rouge et on peut étudier "
    "ses caractéristiques spécifiques.\n\n"
    "• Pourquoi 'nucleus' (noyau) et pas 'cell' (cellule) ? Les échantillons de cerveau post-mortem sont congelés "
    "depuis des années. Les membranes cellulaires sont dégradées, mais les noyaux restent intacts. Le snRNA-seq "
    "extrait donc les noyaux et séquence l'ARNm qu'ils contiennent. Conséquence technique majeure : le pourcentage "
    "de comptages mitochondriaux (pct_counts_mt) est naturellement très faible (~0%), car les mitochondries "
    "(organites cytoplasmiques) ne sont pas capturées avec les noyaux."
)

doc.add_heading('1.3. Description du dataset', level=2)
doc.add_paragraph(
    "Les données analysées proviennent de l'Atlas à l'échelle de la population des troubles croisés du cortex "
    "préfrontal humain (Lee et al., 2024), une ressource transcriptomique majeure accessible via le portail "
    "CellxGene Census (Chan Zuckerberg Initiative). Cet atlas initial profile plus de 6,3 millions de noyaux "
    "provenant de 1 494 donneurs post-mortem couvrant 26 diagnostics neuropsychiatriques distincts. Pour cette "
    "étude, nous avons extrait une cohorte ciblée et rigoureusement filtrée de 62 800 noyaux provenant de "
    "17 donneurs, incluant 8 patients atteints de la maladie d'Alzheimer (AD), 3 patients atteints de la maladie "
    "de Parkinson (PD) et 6 contrôles neurotypiques (CTRL). Cette réduction drastique a été motivée par des "
    "contraintes de pureté analytique (exclusion des comorbidités) et des limitations matérielles (RAM disponible)."
)

add_technical_note(doc, "Problème de RAM et réduction du dataset",
    "Le dataset initial (6,3 millions de noyaux, fichier .h5ad de 6,28 GB) dépasse largement les capacités "
    "mémoire d'une infrastructure de calcul standard (16-32 GB RAM). Le chargement en mémoire vive (adata.to_memory()) "
    "des métriques de QC (calcul de pct_counts_mt, n_genes) provoquait systématiquement des erreurs MemoryError. "
    "Stratégie de contournement adoptée : (1) Exclusion précoce de l'ascendance 'African' (~58% du dataset) "
    "pour réduire le fichier à ~2,5 GB, (2) Lecture en mode backed ('r') limitant les opérations possibles, "
    "(3) Copie en RAM uniquement après filtrage initial. Cette limitation technique majeure a imposé un "
    "sous-échantillonnage massif et constitue un biais méthodologique à discuter (section 4.2)."
)

doc.add_heading('1.4. Question biologique et objectifs', level=2)
doc.add_paragraph(
    "La question biologique centrale de ce projet est :\n\n"
    "« Quelles sont les signatures transcriptomiques cellulaires communes et spécifiques partagées entre la "
    "maladie d'Alzheimer (AD) et la maladie de Parkinson (PD) au niveau du cortex préfrontal dorsolatéral, "
    "et comment le sexe biologique module-t-il ces signatures ? »"
)

doc.add_paragraph("Les objectifs secondaires sont les suivants :")
doc.add_paragraph(
    "1. Identifier et annoter de manière robuste les principaux types cellulaires du DLPFC (neurones, glie) "
    "et quantifier leur composition relative dans les groupes pathologiques versus contrôles.\n\n"
    "2. Détecter les gènes différentiellement exprimés (DEGs) dans chaque type cellulaire, en comparant "
    "AD vs CTRL et PD vs CTRL, tout en contrôlant les facteurs confondants (sexe, ascendance génétique).\n\n"
    "3. Caractériser fonctionnellement les signatures identifiées via des analyses d'enrichissement (Gene Ontology, "
    "GSEA) pour interpréter les mécanismes biologiques sous-jacents.\n\n"
    "4. Évaluer la convergence ou la divergence des altérations transcriptomiques entre AD et PD au niveau "
    "de types cellulaires spécifiques (notamment microglie, astrocytes, neurones excitateurs)."
)

doc.add_heading('1.5. Approche analytique globale', level=2)
doc.add_paragraph(
    "Le pipeline analytique a été structuré en 4 notebooks Jupyter modulaires et séquentiels, combinant des outils "
    "Python (Scanpy 1.9+ pour le prétraitement, clustering et annotation) et R (DESeq2 pour l'analyse différentielle "
    "et clusterProfiler pour l'enrichissement fonctionnel). Le cœur de la stratégie méthodologique repose sur "
    "l'approche pseudobulk, où les comptages de transcrits bruts sont agrégés par donneur et par type cellulaire "
    "avant l'analyse différentielle. Cette méthode corrige le problème de pseudoréplication inhérent aux données "
    "single-cell (où les milliers de cellules d'un même donneur ne sont pas des réplicats biologiques indépendants) "
    "et permet d'utiliser les méthodes statistiques robustes développées pour le bulk RNA-seq (DESeq2, limma/voom)."
)

add_pedagogical_note(doc, "Pipeline modulaire et reproductibilité",
    "Le projet a été organisé en 4 notebooks Jupyter successifs :\n"
    "• Notebook 1 (reduction_of_dataset.ipynb) : QC cellulaire + sélection de cohorte → adata_filtered.h5ad\n"
    "• Notebook 2 (single_cell_pipeline.ipynb) : Normalisation + HVG + PCA + UMAP + Leiden → adata_pp.h5ad\n"
    "• Notebook 3 (add_cell_type_annotation.ipynb) : Annotation manuelle 23 clusters → 8 macro-types → adata_annotated.h5ad\n"
    "• Notebook 4 (translation_to_R.ipynb) : Agrégation pseudobulk + filtres → exports CSV pour R\n\n"
    "Chaque notebook prend en entrée le fichier .h5ad de l'étape précédente, effectue une transformation précise, "
    "et exporte un nouveau fichier pour l'étape suivante. Analogie : Une chaîne de montage automobile où chaque "
    "station (QC, peinture, assemblage) effectue une tâche spécialisée avant de passer le produit à la station suivante. "
    "Cette modularité garantit la reproductibilité, facilite le débogage et permet de relancer le pipeline à partir "
    "de n'importe quelle étape intermédiaire."
)

doc.add_page_break()

# METHODES
doc.add_heading('2. Matériel et Méthodes', level=1)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
text = "[FIGURE 1 À INSÉRER : Pipeline analytique global]"
run = p.add_run(text)
run.font.color.rgb = RGBColor(255, 140, 0)
run.bold = True
run.italic = True
run.font.size = Pt(11)

doc.add_heading('2.1. Sélection et réduction de la cohorte', level=2)
doc.add_paragraph(
    "La cohorte a été réduite du dataset initial de 6,3 millions de noyaux (693 682 après pré-filtrage technique) "
    "à un ensemble analytique de 62 800 noyaux via une stratégie de filtrage en plusieurs étapes."
)

doc.add_heading('2.1.1. Critères d\'inclusion et d\'exclusion', level=3)
doc.add_paragraph(
    "Stratégie de pureté diagnostique : Seuls les donneurs présentant un diagnostic principal unique ont été retenus :"
)
doc.add_paragraph(
    "• Maladie d'Alzheimer pure (dementia || Alzheimer disease) : Exclusion de toute comorbidité (démence vasculaire, "
    "diabète, tauopathies).\n\n"
    "• Maladie de Parkinson pure (dementia || Parkinson disease) : Exclusion des formes mixtes (AD+PD, tauopathies).\n\n"
    "• Contrôles neurotypiques (normal) : Absence de tout diagnostic neuropsychiatrique documenté.\n\n"
    "Filtrage par ascendance génétique :\n"
    "• Exclusion de l'ascendance 'African' (58% du dataset initial, n=408 606 noyaux)\n"
    "• Exclusion de l'ascendance 'unknown' (18,8%, n=130 601 noyaux)\n"
    "• Conservation des ascendances 'European', 'East Asian' et 'Asian' (total : 22,3%, n=154 475 noyaux)"
)

add_technical_note(doc, "Justification controversée de l'exclusion de l'ascendance africaine",
    "Cette décision méthodologique est la plus problématique du projet et mérite une justification transparente.\n\n"
    "Raison pragmatique (contrainte matérielle) : L'exclusion de l'ascendance africaine a réduit le fichier .h5ad "
    "de 6,28 GB à ~2,5 GB, permettant de charger les données en RAM pour le calcul des métriques QC (pct_counts_mt, "
    "n_genes). Sans cette réduction, les opérations produisaient des erreurs MemoryError systématiques sur "
    "l'infrastructure disponible (16-32 GB RAM).\n\n"
    "Conséquence scientifique (biais de sélection majeur) : Nos conclusions ne sont généralisables qu'aux "
    "populations d'ascendance européenne et asiatique. Les signatures moléculaires identifiées pourraient différer "
    "substantiellement dans les populations d'ascendance africaine en raison de variations génétiques, épigénétiques "
    "ou environnementales. Ce biais limite sévèrement la portée translationnelle des résultats et constitue une "
    "limitation éthique et scientifique majeure (voir Discussion, section 4.2).\n\n"
    "Alternative non retenue : L'utilisation de ressources de calcul haute performance (HPC, cloud computing) "
    "aurait permis d'éviter ce biais, mais n'était pas accessible dans le cadre pédagogique du projet."
)

doc.add_heading('2.1.2. Caractéristiques de la cohorte finale', level=3)

# --- INSERTION NOTE COMPARATIVE ---
add_methodo_note(doc, "Justification de la réduction d'effectif (Stratégie QC)",
    "Lors de la constitution de la cohorte, deux approches ont été comparées :\n"
    "1. Une sélection basée uniquement sur les métadonnées (Diagnostic/Ascendance) aboutit à ~77 000 noyaux.\n"
    "2. Notre approche, qui applique un filtrage qualité immédiat (n_genes > 1000, mt% < 5%), aboutit à ~68 800 noyaux.\n\n"
    "Nous avons retenu cette seconde approche. Bien que réduisant l'effectif d'environ 11%, elle élimine les "
    "noyaux de faible qualité technique (débris, stress cellulaire) qui auraient introduit du bruit dans "
    "le clustering et l'analyse différentielle."
)

doc.add_paragraph(
    "La cohorte finale comprend 17 donneurs post-mortem et 62 800 noyaux. Le Tableau 1 résume les caractéristiques "
    "démographiques et cellulaires par groupe diagnostique."
)

data_table1 = [
    ['Alzheimer (AD)', '8', '25 301', '3 163', '68% / 32%', 'Européenne (37,5%), East Asian (50%), Asian (12,5%)'],
    ['Parkinson (PD)', '3', '7 023', '2 341', '73% / 27%', 'Européenne (33,3%), East Asian (66,7%)'],
    ['Normal (CTRL)', '6', '30 476', '5 079', '48,5% / 51,5%', 'Européenne (100%)']
]
add_table_with_style(doc, data_table1, 
    ['Diagnostic', 'N Donneurs', 'N Noyaux', 'Noyaux/Donneur (moy.)', 'Sexe (M/F %)', 'Ascendance génétique'],
    'Tableau 1. Caractéristiques démographiques et cellulaires de la cohorte finale (n=17 donneurs, 62 800 noyaux)'
)

add_pedagogical_note(doc, "Déséquilibres démographiques de la cohorte et implications statistiques",
    "Trois déséquilibres majeurs doivent être notés :\n\n"
    "1. Faible taille du groupe PD (n=3 donneurs) : Limite sévèrement la puissance statistique pour détecter des DEGs "
    "dans ce groupe. Risque élevé de faux négatifs (vrais effets biologiques non détectés). Conséquence : Les résultats "
    "pour PD doivent être interprétés avec une extrême prudence.\n\n"
    "2. Biais de sexe : Les groupes AD (68% hommes) et PD (73% hommes) sont fortement masculins, alors que le groupe "
    "CTRL est équilibré (48,5% hommes). Le sexe biologique étant un facteur confondant majeur dans l'expression génique "
    "cérébrale, il DOIT être inclus comme covariable dans le modèle DESeq2 (voir section 2.6).\n\n"
    "3. Biais d'ascendance : Le groupe CTRL est 100% européen, alors que AD et PD sont mixtes (européen + asiatique). "
    "L'ascendance génétique influence l'expression génique de base. Elle DOIT également être contrôlée dans le modèle "
    "statistique pour isoler l'effet pur de la pathologie."
)

add_figure(doc, 
    os.path.join(PATHS["EDA"], "1_disease_distribution.png"),
    "Figure 2. Distribution des noyaux par groupe diagnostique (AD : 25 301, PD : 7 023, CTRL : 30 476)",
    width=Inches(5.5)
)

doc.add_heading('2.2. Contrôle qualité (QC) et filtration cellulaire', level=2)
doc.add_paragraph(
    "Les métriques de qualité cellulaire suivantes ont été calculées sur les comptages bruts (avant normalisation) :"
)
doc.add_paragraph(
    "• n_genes_by_counts : Nombre de gènes détectés par noyau (au moins 1 comptage UMI)\n"
    "• n_counts (total_counts) : Nombre total de transcrits UMI capturés par noyau (profondeur de séquençage)\n"
    "• pct_counts_mt : Pourcentage de comptages mitochondriaux (indicateur de stress cellulaire ou mort)\n\n"
    "Seuils de filtration appliqués (basés sur les distributions observées, Figure 3) :\n"
    "• 1 000 ≤ n_genes_by_counts ≤ 6 000\n"
    "• pct_counts_mt ≤ 5%\n\n"
    "Résultat du filtrage QC : 285 076 noyaux (pré-filtrés) → 243 011 noyaux (post-QC) → Rétention de 85,3%"
)

add_pedagogical_note(doc, "Interprétation des seuils de qualité",
    "• n_genes < 1 000 : Noyau vide, débris cellulaires ou capture technique échouée. Ces objets ne contiennent "
    "pas assez d'information biologique exploitable.\n\n"
    "• n_genes > 6 000 : Probable doublet (deux noyaux collés comptés comme un seul objet). Les doublets faussent "
    "l'analyse en créant des 'cellules artificielles' avec un profil d'expression hybride.\n\n"
    "• pct_counts_mt > 5% : Dans le bulk RNA-seq classique, des valeurs élevées (>10-20%) indiquent des cellules "
    "stressées ou apoptotiques (fuite de l'ARN nucléaire, enrichissement relatif des transcrits mitochondriaux). "
    "ATTENTION : En snRNA-seq (séquençage de noyaux), pct_counts_mt est naturellement très faible (~0%) car les "
    "mitochondries (organites cytoplasmiques) ne sont PAS capturées avec les noyaux. Un seuil strict à 5% élimine "
    "les rares contaminations cytoplasmiques."
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES"], "qc_violin_plots.png"),
    "Figure 3. Distributions des métriques de qualité cellulaire avant filtrage (n=285 076 noyaux). "
    "Les lignes rouges indiquent les seuils de filtration appliqués.",
    width=Inches(6.5)
)

doc.add_heading('2.3. Normalisation et sélection de gènes hautement variables', level=2)
doc.add_paragraph(
    "Les étapes de transformation suivantes ont été appliquées dans l'ordre :"
)
doc.add_paragraph(
    "1. Sauvegarde des comptages bruts : Les comptages UMI bruts ont été préservés dans adata.layers['counts'] "
    "AVANT toute transformation. Cette étape est critique pour l'analyse différentielle ultérieure (DESeq2 exige "
    "des comptages entiers non normalisés).\n\n"
    "2. Normalisation par profondeur de séquençage : Chaque noyau a été normalisé à 10 000 UMI totaux "
    "(scanpy.pp.normalize_total, target_sum=1e4) pour corriger les différences de profondeur de séquençage entre noyaux.\n\n"
    "3. Log-transformation : Application de log(x+1) (scanpy.pp.log1p) pour stabiliser la variance et rapprocher "
    "la distribution des comptages d'une distribution normale, condition requise pour les méthodes linéaires (PCA).\n\n"
    "4. Sélection des gènes hautement variables (HVGs) : Identification et conservation des 2 000 gènes présentant "
    "la plus forte variabilité biologique entre noyaux (méthode Seurat v3, scanpy.pp.highly_variable_genes). "
    "Cette étape réduit le bruit technique et la charge computationnelle tout en préservant le signal biologique."
)

add_technical_note(doc, "Importance critique de la préservation des raw counts",
    "DESeq2 (et toutes les méthodes d'analyse différentielle basées sur la loi binomiale négative) EXIGENT des "
    "comptages entiers bruts, NON normalisés et NON log-transformés. Pourquoi ?\n\n"
    "Le modèle statistique de DESeq2 modélise la distribution des comptages comme suit :\n"
    "Comptages ~ NegativeBinomial(μ, dispersion)\n"
    "où μ dépend de la taille de la bibliothèque (size factor) et de l'expression du gène.\n\n"
    "Si on fournit des données déjà normalisées ou log-transformées, ce modèle est mathématiquement invalide "
    "(les valeurs ne sont plus des entiers, la variance n'est plus proportionnelle à la moyenne). Conséquence : "
    "Les p-values et les estimations de fold-change deviennent incorrectes.\n\n"
    "C'est pourquoi adata.layers['counts'] doit impérativement contenir les comptages bruts originaux."
)

doc.add_heading('2.4. Réduction de dimensionnalité et clustering', level=2)
doc.add_paragraph(
    "Réduction de dimensionnalité par PCA :\n"
    "Une Analyse en Composantes Principales (PCA) a été calculée sur les 2 000 HVGs (50 composantes principales, "
    "scanpy.tl.pca). Les 30 premières PCs, capturant la majorité de la variance biologique (Figure 4), ont été "
    "retenues pour les étapes suivantes.\n\n"
    "Construction du graphe de voisinage :\n"
    "Un graphe k-nearest neighbors (kNN) a été construit dans l'espace PCA à 30 dimensions (scanpy.pp.neighbors, "
    "n_neighbors=15). Ce graphe représente la similarité transcriptomique entre noyaux : deux noyaux sont connectés "
    "s'ils figurent parmi les 15 voisins les plus proches l'un de l'autre.\n\n"
    "Clustering Leiden :\n"
    "L'algorithme de détection de communautés Leiden (résolution=0.5, scanpy.tl.leiden) a été appliqué au graphe "
    "de voisinage pour partitionner les 62 800 noyaux en 23 clusters initiaux. Chaque cluster représente un "
    "groupe de noyaux transcriptomiquement homogènes, potentiellement correspondant à un type/sous-type cellulaire "
    "ou à un état fonctionnel distinct.\n\n"
    "Visualisation UMAP :\n"
    "Une projection UMAP (Uniform Manifold Approximation and Projection, scanpy.tl.umap) a été générée pour "
    "visualiser la structure globale du dataset en 2 dimensions (Figure 5)."
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_SCP"], "pca_variance_ratio.pdf"),
    "Figure 4. Variance expliquée par les 50 premières composantes principales (PCA). Les 30 premières PCs "
    "ont été retenues pour la construction du graphe de voisinage.",
    width=Inches(5.5)
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_SCP"], "umap_leiden.pdf"),
    "Figure 5. Projection UMAP des 62 800 noyaux colorés par cluster Leiden (23 clusters identifiés, résolution=0.5). "
    "Les clusters sont spatialement séparés, indiquant une bonne qualité de clustering.",
    width=Inches(6)
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_RAPPORT"], "pca_overview.pdf"),
    "Figure 6. Vue d'ensemble de l'analyse PCA : scatter plot PC1/PC2, variance expliquée, et loadings des top gènes.",
    width=Inches(6.5)
)

doc.add_heading('2.5. Annotation des types cellulaires', level=2)
doc.add_paragraph(
    "Les 23 clusters Leiden ont été annotés manuellement en 8 macro-types cellulaires basés sur l'expression "
    "de marqueurs canoniques de la littérature (Tableau 2). Cette annotation a été réalisée via l'inspection "
    "des gènes différentiellement exprimés par cluster (test de Wilcoxon, scanpy.tl.rank_genes_groups) et "
    "comparée aux atlas de référence (PanglaoDB, CellMarker)."
)

data_table2 = [
    ['Neurones excitateurs', '0, 2, 8, 11, 13, 16, 19', '22 769 (36,3%)', 'SLC17A7, SATB2, TBR1'],
    ['Neurones inhibiteurs', '1, 7, 12, 20', '13 028 (20,7%)', 'GAD1, GAD2, PVALB, SST, VIP'],
    ['Oligodendrocytes/OPC', '3, 6, 14', '9 696 (15,4%)', 'MBP, MOG, OLIG1, PDGFRA'],
    ['Astrocytes Homéostatiques', '4', '4 752 (7,6%)', 'AQP4, GFAP, SLC1A2'],
    ['Astrocytes Réactifs', '15', '685 (1,1%)', 'GFAP, VIM, SERPINA3'],
    ['Microglie', '5, 18 (fusionnés)', '4 875 (7,8%)', 'CX3CR1, P2RY12, TMEM119, APOE, TREM2'],
    ['Support/Vascular/Immune', '9, 10, 17', '6 611 (10,5%)', 'CLDN5, PDGFRB, CD3E'],
    ['Unknown/Low Quality', '21, 22 (exclus)', '384 (0,6%)', 'Profil mixte ou dégradé']
]
add_table_with_style(doc, data_table2,
    ['Macro-type cellulaire', 'Clusters Leiden', 'N Noyaux (%)', 'Marqueurs canoniques'],
    'Tableau 2. Annotation manuelle des 23 clusters Leiden en 8 macro-types cellulaires. Les clusters 21-22 '
    '(Unknown/Low Quality) ont été exclus de l\'analyse pseudobulk finale.'
)

add_technical_note(doc, "Fusion stratégique du Cluster 18 (Microglie DAM) et validation pseudobulk",
    "Problème identifié : Le Cluster 18, initialement annoté comme 'Microglie DAM' (Disease-Associated Microglia, "
    "signature APOE+/TREM2+/P2RY12-), contenait 393 noyaux provenant à 98,2% d'un SEUL donneur (Donor_865). "
    "Les 16 autres donneurs contribuaient à peine 7 noyaux au total.\n\n"
    "Décision méthodologique : Fusion avec le Cluster 5 (Microglie Homéostatique) pour former un macro-type "
    "'Microglie' unifié (n=4 875 noyaux, 15 donneurs avec ≥20 cellules). Justification : L'approche pseudobulk "
    "exige un minimum de 5 donneurs par type cellulaire pour garantir des réplicats biologiques suffisants et "
    "éviter les artefacts individuels. Avec un seul donneur dominant, l'analyse différentielle sur 'Microglie DAM' "
    "aurait été statistiquement invalide (confondre effet pathologique et variation inter-individuelle).\n\n"
    "Conséquence biologique : Perte de granularité sur l'état d'activation microglial. Les signatures DAM "
    "(inflammation, phagocytose) seront diluées dans le macro-type 'Microglie' unifié, réduisant potentiellement "
    "la sensibilité pour détecter des gènes spécifiques à l'activation pathologique."
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_SCP"], "markers_dotplot.pdf"),
    "Figure 7. Expression des gènes marqueurs canoniques à travers les 23 clusters Leiden. La taille des points "
    "indique le pourcentage de noyaux exprimant le gène ; la couleur indique le niveau d'expression moyen (échelle log).",
    width=Inches(6.5)
)

doc.add_heading('2.6. Agrégation pseudobulk et analyse différentielle', level=2)
doc.add_paragraph(
    "Principe de l'approche pseudobulk :\n"
    "Les comptages UMI bruts (adata.layers['counts']) ont été agrégés (somme) par combinaison unique de "
    "(type_cellulaire, donneur), créant des 'échantillons pseudobulk'. Par exemple, pour le type 'Microglie', "
    "les comptages de tous les noyaux microglials du Donor_865 ont été sommés pour créer un échantillon "
    "'Microglia__Donor_865'. Cette agrégation transforme les données single-cell en format bulk, où chaque "
    "donneur devient un réplicat biologique indépendant.\n\n"
    "Filtres de robustesse statistique appliqués :\n"
    "1. MIN_CELLS_PER_SAMPLE ≥ 20 : Exclusion des échantillons pseudobulk contenant moins de 20 noyaux "
    "(agrégation insuffisante, comptages trop faibles)\n"
    "2. MIN_DONORS_PER_CLUSTER ≥ 5 : Exclusion des types cellulaires présents chez moins de 5 donneurs "
    "(réplication biologique insuffisante)\n"
    "3. Exclusion manuelle du macro-type 'Unknown/Low Quality' (noyaux de mauvaise qualité, profils dégradés)\n\n"
    "Résultat final : 96 échantillons pseudobulk couvrant 7 types cellulaires robustes (Excitatory neuron, "
    "Inhibitory neuron, Oligodendrocyte/OPC, Astrocyte Homeostatic, Astrocyte Reactive, Microglia, "
    "Support/Vascular/Immune)."
)

doc.add_paragraph(
    "Modèle d'analyse différentielle (DESeq2) :\n"
    "L'analyse DGE a utilisé le package R DESeq2 (Love et al., 2014) avec le modèle linéaire généralisé suivant :\n\n"
    "Expression ~ sex + genetic_ancestry + disease\n\n"
    "où :\n"
    "• sex : Sexe biologique (male/female) — covariable d'ajustement\n"
    "• genetic_ancestry : Ascendance génétique (European/East Asian/Asian) — covariable d'ajustement\n"
    "• disease : Groupe diagnostique (AD/PD/CTRL) — variable d'intérêt\n\n"
    "Contrastes testés :\n"
    "• AD vs CTRL (pour chaque type cellulaire)\n"
    "• PD vs CTRL (pour chaque type cellulaire)\n\n"
    "Seuils de significativité :\n"
    "• p-value ajustée (FDR, méthode Benjamini-Hochberg) : padj < 0,05\n"
    "• Fold-change biologique minimum : |log2FC| > 0,5 (soit FC > 1,41 ou FC < 0,71)"
)

add_pedagogical_note(doc, "Rôle des covariables d'ajustement dans le modèle DESeq2",
    "Pourquoi inclure 'sex' et 'genetic_ancestry' dans le modèle ?\n\n"
    "Sans ajustement, si les groupes AD et CTRL diffèrent systématiquement en termes de sexe (ex: AD=70% hommes, "
    "CTRL=50% hommes), les DEGs identifiés pourraient refléter des différences liées au SEXE plutôt qu'à la PATHOLOGIE. "
    "Le modèle DESeq2 'nettoie' statistiquement l'effet du sexe et de l'ascendance AVANT de calculer l'effet de "
    "la maladie. On obtient ainsi l'effet 'pur' de AD vs CTRL, indépendant des biais démographiques.\n\n"
    "Analogie : Imaginez mesurer l'effet d'un médicament sur la tension artérielle, mais vos groupes traité/placebo "
    "ont des âges différents. Vous devez 'contrôler pour l'âge' dans votre analyse statistique pour isoler l'effet "
    "réel du médicament."
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_SCP"], "umap_diagnosis.pdf"),
    "Figure 8. Projection UMAP des 62 800 noyaux colorés par groupe diagnostique (AD/PD/CTRL). Le bon mélange "
    "spatial des conditions indique l'absence d'effets batch techniques majeurs entre groupes.",
    width=Inches(6)
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES_RAPPORT"], "umap_genetic_ancestry.pdf"),
    "Figure 9. Projection UMAP colorée par ascendance génétique. La répartition spatiale confirme l'hétérogénéité "
    "de la cohorte et justifie l'inclusion de cette variable dans le modèle statistique.",
    width=Inches(6)
)

doc.add_page_break()

# RÉSULTATS
doc.add_heading('3. Résultats', level=1)

doc.add_heading('3.1. Caractérisation globale de la cohorte', level=2)
doc.add_paragraph(
    "La projection UMAP de la cohorte finale (Figure 8) montre un bon mélange spatial des trois groupes "
    "diagnostiques (AD, PD, CTRL), sans ségrégation évidente par condition. Cette observation suggère que les "
    "différences transcriptomiques entre groupes sont subtiles et spécifiques à certains types cellulaires, "
    "plutôt que globales et systématiques. L'absence de clustering par donneur ou par batch technique confirme "
    "l'efficacité de la normalisation et l'absence d'artefacts batch majeurs. La Figure 9 confirme également "
    "un bon mélange des ascendances génétiques, validant la stratégie de contrôle dans le modèle statistique."
)

doc.add_heading('3.2. Composition cellulaire du DLPFC', level=2)
doc.add_paragraph(
    "Les 8 macro-types cellulaires identifiés montrent la composition attendue du cortex cérébral, "
    "dominée par les neurones (57% du total : 36,3% excitateurs + 20,7% inhibiteurs) et complétée par les "
    "populations gliales (oligodendrocytes/OPC 15,4%, astrocytes 8,7%, microglie 7,8%). La composition "
    "relative est stable entre groupes diagnostiques, suggérant que les altérations pathologiques sont "
    "principalement transcriptomiques plutôt que démographiques (pas de perte massive de types cellulaires)."
)

add_figure(doc, 
    os.path.join(PATHS["FIGURES"], "umap_precomputed_class.png"),
    "Figure 10. Projection UMAP colorée par macro-type cellulaire (classification 'class'). Les neurones excitateurs "
    "(EN, rouge) et inhibiteurs (IN, bleu) dominent numériquement, mais tous les types cellulaires majeurs du cortex "
    "sont représentés et spatialement séparés.",
    width=Inches(6)
)

add_figure(doc, 
    os.path.join(PATHS["EDA"], "2_cell_types_distribution.png"),
    "Figure 11. Distribution des noyaux par macro-type cellulaire. Les oligodendrocytes (Oligo) et neurones excitateurs "
    "(EN) dominent numériquement le cortex préfrontal.",
    width=Inches(5.5)
)

doc.add_heading('3.3. Gènes différentiellement exprimés (DEGs)', level=2)

doc.add_paragraph(
    "⚠️ AVERTISSEMENT MÉTHODOLOGIQUE ⚠️\n\n"
    "Les résultats quantitatifs présentés ci-dessous (nombres de DEGs, log2FC, padj) sont basés sur une analyse "
    "R/DESeq2 complète mais DOIVENT être considérés comme préliminaires. L'interprétation biologique (signatures "
    "inflammatoires, dysfonction synaptique) est robuste et cohérente avec la littérature, mais les valeurs "
    "numériques exactes sont limitées par la taille réduite de la cohorte, en particulier pour la maladie de Parkinson (PD, n=3).\n\n"
    "Le Tableau 3 résume les résultats de l'analyse différentielle par type cellulaire et contraste, en distinguant "
    "les gènes significatifs après correction pour tests multiples (FDR<0,05) et les gènes en forte tendance "
    "(P<0,01) qui restent biologiquement informatifs malgré l'absence de significativité stricte."
)

# Tableau 3 depuis CSV
if os.path.exists(PATHS["R_SUMMARY"]):
    try:
        summary_df = pd.read_csv(PATHS["R_SUMMARY"])
        table3_data = []
        for _, row in summary_df.iterrows():
            cell_type = row.get("Cell Type", row.get("File", "").split(".")[0])
            contrast = row.get("Contrast", "")
            total_genes = row.get("Total Genes", "")
            trends = row.get("Trends (P<0.01)", row.get("Trends", ""))
            signif = row.get("Significant (FDR<0.05)", row.get("Signif", ""))
            top_genes = row.get("Top Genes", "")
            table3_data.append([cell_type, contrast, total_genes, trends, signif, top_genes])
        
        # Nettoyage chiffres
        cleaned_data = []
        for row in table3_data:
            cleaned_row = []
            for item in row:
                try:
                    val = float(item)
                    if val.is_integer(): cleaned_row.append(str(int(val)))
                    else: cleaned_row.append(str(val))
                except:
                    cleaned_row.append(str(item))
            cleaned_data.append(cleaned_row)

        add_table_with_style(
            doc,
            cleaned_data,
            ['Cell type', 'Contraste', 'Total', 'Tendances (P<0.01)', 'Signif. (FDR<0.05)', 'Top gènes'],
            "Tableau 3. Résumé des résultats d'expression différentielle (analyse pseudobulk) par type cellulaire et contraste."
        )
    except Exception as e:
        doc.add_paragraph(f"Erreur tableau : {e}")

doc.add_paragraph(
    "Observations principales (basées sur cette analyse) :\n"
    "• Le cluster Support/Vascular/Immune présente le signal le plus robuste, avec au moins un gène significatif "
    "(FDR<0,05) et plusieurs gènes en forte tendance (P<0,01), suggérant une perturbation de la barrière "
    "hémato-encéphalique et de l'environnement vasculaire dans l'AD.\n"
    "• La microglie montre un ensemble de gènes en tendance mais aucun gène ne franchit le seuil de FDR<0,05 "
    "dans cette cohorte réduite, illustrant la forte hétérogénéité inter-individuelle des réponses inflammatoires.\n"
    "• Dans la maladie de Parkinson, aucun gène n'est significatif après correction, mais plusieurs types "
    "cellulaires (notamment les astrocytes) présentent des gènes en tendance (P<0,01), compatibles avec "
    "une activation gliale plus modeste."
)

# INSERTION AUTOMATIQUE IMAGES R
add_figure(
    doc,
    os.path.join(PATHS["R_VOLCANO"], "Volcano_Support.Vascular.Immune_AD_vs_CTRL.png"),
    "Figure 12. Volcano plot du cluster Support/Vascular/Immune (AD vs CTRL). "
    "Le point rouge identifie le lncRNA MEG3 significativement sous-exprimé (FDR < 0,05), "
    "tandis que les points oranges (P < 0,01) correspondent à des gènes montrant une forte tendance biologique.",
    width=Inches(5.5)
)

add_figure(
    doc,
    os.path.join(PATHS["R_BAR"], "Barplot_Support.Vascular.Immune_AD_vs_CTRL.png"),
    "Figure 13. Top 15 des gènes les plus dérégulés (|log2FC|) dans le cluster Support/Vascular/Immune (AD vs CTRL). "
    "Les barres rouges indiquent les gènes surexprimés, les barres bleues ceux sous-exprimés, avec MEG3 parmi "
    "les gènes les plus fortement diminués.",
    width=Inches(6)
)

doc.add_heading('3.3.1. Signatures microgliales', level=3)
doc.add_paragraph(
    "Dans la microglie, l'analyse stricte (FDR<0,05) ne met en évidence aucun gène significatif dans cette cohorte. "
    "Cependant, l'analyse des tendances (P<0,01) révèle un ensemble restreint de gènes candidats suggérant une "
    "réorganisation transcriptionnelle subtile compatible avec une activation immunitaire hétérogène. Parmi ces gènes, "
    "on note notamment le facteur de transcription POU3F2, ainsi que plusieurs gènes impliqués dans la signalisation "
    "et la régulation transcriptionnelle."
)

add_pedagogical_note(doc, "Pourquoi visualiser les tendances (P<0,01) ?",
    "Avec une cohorte de taille limitée (en particulier pour PD), la correction pour tests multiples (FDR) est très "
    "sévère et peut conduire à un nombre nul de gènes significatifs, même en présence de signaux biologiques réels. "
    "La visualisation des gènes avec P<0,01 (sans correction) permet d'identifier des 'tendances' cohérentes qui "
    "peuvent être discutées comme des pistes exploratoires, tout en restant transparent sur l'absence de "
    "significativité stricte."
)

add_figure(
    doc,
    os.path.join(PATHS["R_VOLCANO"], "Volcano_Microglia_AD_vs_CTRL.png"),
    "Figure 14. Volcano plot de l'analyse différentielle Microglie (AD vs CTRL). "
    "L'absence de points rouges (FDR<0,05) illustre la faible puissance statistique et l'hétérogénéité "
    "inter-individuelle, tandis que la zone orange (P<0,01) met en évidence un ensemble de gènes candidats "
    "potentiellement impliqués dans la réponse microgliale.",
    width=Inches(5.5)
)

# Placeholder restant pour Heatmap
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
text = "[FIGURE 15 À INSÉRER : Heatmap Top DEGs neuronaux]"
run = p.add_run(text)
run.font.color.rgb = RGBColor(255, 140, 0)
run.bold = True
run.italic = True
run.font.size = Pt(11)

# DISCUSSION
doc.add_heading('4. Discussion', level=1)
doc.add_heading('4.1. Synthèse des résultats principaux et convergence mécanistique', level=2)
doc.add_paragraph(
    "Cette étude transcriptomique à résolution unicellulaire du cortex préfrontal dorsolatéral (DLPFC) identifie "
    "deux axes mécanistiques convergents dans la maladie d'Alzheimer (AD), cohérents avec un corpus robuste de "
    "littérature snRNA-seq indépendante :\n\n"
    "1. Neuroinflammation vasculaire et microgliale : La dérégulation du lncRNA MEG3 dans les cellules "
    "Support/Vascular/Immune, associée à des gènes en tendance impliqués dans l'angiogenèse et l'activation "
    "endothéliale, suggère une altération de la barrière hémato-encéphalique. En parallèle, la microglie montre "
    "un ensemble de gènes candidats compatibles avec une activation inflammatoire hétérogène, bien que la plupart "
    "ne franchissent pas le seuil de FDR<0,05 dans cette cohorte.\n\n"
    "2. Dysfonction synaptique et métabolique neuronale : La sous-expression systématique de gènes impliqués dans "
    "la transmission glutamatergique, la fonction mitochondriale et la plasticité neuronale dans les neurones "
    "excitateurs est cohérente avec l'atrophie synaptique et la perte de fonction cognitive décrites dans l'AD.\n\n"
    "Les résultats pour la maladie de Parkinson (PD) sont plus modestes en amplitude et nécessitent une interprétation "
    "prudente en raison de la très faible taille d'échantillon (n=3 donneurs). Les signatures gliales observées "
    "en tendance (notamment dans les astrocytes) suggèrent une neuroinflammation partagée mais d'intensité plus faible."
)

doc.add_heading('4.2. Limitations critiques et biais méthodologiques', level=2)
doc.add_paragraph(
    "Cette étude présente plusieurs limitations majeures qui restreignent la portée et la généralisabilité des conclusions :"
)
doc.add_paragraph(
    "1. Biais démographique critique (exclusion de 58% du dataset initial, ascendance africaine) :\n"
    "L'exclusion de la majorité des données d'ascendance africaine pour des raisons de contraintes matérielles (RAM) "
    "constitue un biais de sélection éthiquement et scientifiquement problématique. Conséquences multiples :\n"
    "• Validité externe limitée : Les conclusions ne sont généralisables qu'aux populations d'ascendance européenne "
    "et asiatique (~22% de la population mondiale).\n"
    "• Perpétuation des inégalités en recherche biomédicale : Les populations africaines sont déjà sous-représentées "
    "dans la recherche génomique et transcriptomique (< 3% des études GWAS). Cette exclusion aggrave ce déséquilibre.\n"
    "• Risque de biais moléculaire : Les variants génétiques influençant l'expression génique (eQTLs) diffèrent entre "
    "populations. Les polymorphismes de APOE (notamment la fréquence réduite de l'allèle APOE4 en Afrique subsaharienne) "
    "pourraient moduler différemment les signatures inflammatoires et vasculaires.\n\n"
    "2. Puissance statistique insuffisante pour PD (n=3 donneurs vs n=8 AD, n=6 CTRL) :\n"
    "Avec seulement 3 réplicats biologiques, le risque de faux négatifs (Type II error) est élevé. Les effets "
    "biologiques de magnitude modérée ne peuvent être détectés de manière fiable. L'absence de DEGs significatifs "
    "dans certains types cellulaires pour PD vs CTRL ne peut donc pas être interprétée comme 'absence d'effet', "
    "mais comme 'manque de puissance'.\n\n"
    "3. Métadonnées cliniques manquantes (âge, PMI, stade pathologique, génotype APOE) :\n"
    "L'absence de ces covariables empêche de distinguer l'effet de la pathologie de celui du vieillissement ou "
    "d'autres facteurs confondants majeurs.\n\n"
    "4. Perte de résolution biologique (fusion de sous-types, agrégation de macro-types) :\n"
    "La fusion de sous-populations microgliales et l'agrégation de sous-types neuronaux excitateurs réduisent "
    "la capacité à identifier des vulnérabilités très spécifiques à certains sous-types.\n\n"
    "5. Limitation temporelle (données post-mortem transversales) :\n"
    "Les données représentent un instantané terminal de la pathologie et ne permettent pas de reconstruire "
    "directement la dynamique temporelle des événements moléculaires."
)

add_technical_note(doc, "LIMITATION CRITIQUE — Absence de l'âge dans le modèle statistique",
    "L'âge est LE facteur de risque majeur pour AD et PD. En l'absence de la variable 'age_at_death' dans les "
    "métadonnées, il est impossible d'ajuster explicitement pour ce facteur dans le modèle DESeq2. Par conséquent, "
    "les signatures identifiées doivent être interprétées comme des associations plutôt que comme des preuves "
    "causales strictes. Une validation sur des cohortes disposant d'informations cliniques complètes (ROSMAP, ADNI) "
    "sera nécessaire pour raffiner ces conclusions."
)

# PERSPECTIVES ET ANALYSES FUTURES
doc.add_heading('4.3. Perspectives et analyses futures', level=2)

doc.add_paragraph(
    "Plusieurs axes d'amélioration biologique et analytique sont envisageables pour approfondir ces résultats :\n\n"
    "1. Analyse de sous-types cellulaires fins (high-resolution clustering), notamment pour identifier les sous-populations "
    "de microglies (ex: états prolifératifs vs inflammatoires) invisibles au niveau macro-type.\n\n"
    "2. Analyse de trajectoires temporelles (Pseudotime / RNA Velocity) pour reconstruire la dynamique "
    "d'activation microgliale et la transition progressive des états neuronaux.\n\n"
    "3. Intégration multi-omique : Croisement avec des données d'épigénomique (ATAC-seq) pour valider les facteurs "
    "de transcription candidats (ex: POU3F2) identifiés dans nos tendances.\n\n"
    "4. Analyse de réseaux de co-expression (WGCNA) pour identifier des modules de gènes co-régulés "
    "et des 'hub genes' potentiels cibles thérapeutiques."
)

# Note méthodologique sur le sous-échantillonnage (On garde car c'est très fort)
add_methodo_note(doc, "Optimisation Stratégique : Le Paradoxe Donneur/Cellule",
    "Une critique majeure de notre approche actuelle est le filtrage drastique des donneurs (N=17) pour satisfaire "
    "la contrainte de RAM, tout en gardant une très haute résolution cellulaire (>4000 cellules/donneur). "
    "Or, la puissance statistique des tests Pseudobulk dépend du nombre de donneurs (N), pas des cellules.\n\n"
    "Perspective prioritaire : Une stratégie de 'Balanced Subsampling' (ex: 500 cellules/donneur) aurait permis "
    "d'inclure plus de 100 donneurs (y compris la cohorte africaine et les cas mixtes) dans la même mémoire RAM. "
    "Cette approche sacrifierait la résolution des sous-types rares pour gagner massivement en puissance statistique."
)

# CONCLUSION
doc.add_heading('4.4. Conclusion générale', level=2)

doc.add_paragraph(
    "Malgré des limitations méthodologiques liées aux ressources de calcul locales, cette analyse transcriptomique "
    "unicellulaire du DLPFC a permis de construire un pipeline robuste et reproductible, allant du traitement brut "
    "à l'interprétation biologique.\n\n"
    "Nos résultats mettent en évidence des signatures biologiquement cohérentes : une altération vasculaire majeure "
    "associée au lncRNA MEG3 (FDR < 0.05) et des tendances inflammatoires microgliales prometteuses. "
    "Ces découvertes valident la pertinence de l'approche 'Pseudobulk' pour éviter les faux positifs inhérents "
    "au single-cell.\n\n"
    "Ce projet constitue donc une preuve de concept solide. L'optimisation de la stratégie d'échantillonnage "
    "(via subsampling) est désormais la prochaine étape pour transformer ces tendances biologiques en "
    "certitudes statistiques généralisables."
)

doc.add_page_break()

# REFERENCES
doc.add_heading('5. Références', level=1)
# (Liste références abrégée ici pour la lisibilité, mais complète dans votre output)
references = [
    "Lee, D. et al. (2024). Population-scale cross-disorder atlas of the human prefrontal cortex. medRxiv.",
    "Wolf, F. A., Angerer, P., & Theis, F. J. (2018). SCANPY: large-scale single-cell gene expression data analysis.",
    "Love, M. I., Huber, W., & Anders, S. (2014). Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2.",
    "Mathys, H. et al. (2019). Single-cell transcriptomic analysis of Alzheimer's disease. Nature.",
]
for i, ref in enumerate(references, 1):
    p = doc.add_paragraph(f"{i}. {ref}", style='Normal')
    p.paragraph_format.left_indent = Inches(0.5)
    p.paragraph_format.first_line_indent = Inches(-0.5)

# SAUVEGARDE
output_dir = os.path.join(BASE_PATH, "rapport")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "Rapport_M2_AIDA_Final_Complet.docx")

try:
    doc.save(output_path)
    print(f"\n✅ SUCCÈS : Rapport sauvegardé dans {output_path}")
except PermissionError:
    print(f"\n❌ ERREUR : Fermez le fichier Word et relancez !")

Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures\single_cell_pipeline\pca_variance_ratio.pdf: 
Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures\single_cell_pipeline\umap_leiden.pdf: 
Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures_pour_rapport\pca_overview.pdf: 
Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures\single_cell_pipeline\markers_dotplot.pdf: 
Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures\single_cell_pipeline\umap_diagnosis.pdf: 
Erreur image C:/Z/AIDA_transcriptomics_project/transcriptomics-code\figures_pour_rapport\umap_genetic_ancestry.pdf: 

✅ SUCCÈS : Rapport sauvegardé dans C:/Z/AIDA_transcriptomics_project/transcriptomics-code\rapport\Rapport_M2_AIDA_Final_Complet.docx
